# LoRA training workflow

Prerequisites: a prepared environment (`torch`, `datasets`, `transformers` with Qwen3 and `dtype` support, `peft`, `accelerate`, and `lm_eval[hf]` for evaluation), the seed-42 HaluEval split from [01_importing_data.ipynb](01_importing_data.ipynb), and Qwen3 model/tokenizer files staged in this project's Hugging Face cache. Full evaluation also needs cached TruthfulQA data. This notebook uses offline loading and installs nothing.

Run **A → B → C** in a fresh kernel. Then choose **D** for the smoke test, **E** for fresh training, or **F → G** for checkpoint inspection/resumption. Run **H** only after a final adapter is saved. D, E, G (when resuming), and H require an existing Unity Slurm GPU allocation; A, B, C, and F require no GPU. Do not launch full tokenization/training on a login node.

All execution controls default to disabled. For the first attempt, enable only `RUN_SMOKE_TEST`; after reviewing it, disable smoke testing and choose `TRAINING_MODE="fresh"`. Choose `"resume"` only after an interrupted run, never alongside fresh training. After editing controls, rerun A, then B and C before the selected branch. GPU objects are local to each training call and cleaned up afterward. Re-running a branch refuses to overwrite populated result/adapter directories.

**Release copy: historical outputs and execution counts have been cleared. This code does not validate any historical run.** A kernel restart is required if Hugging Face libraries were previously imported with different cache settings.


## A — Imports, paths, parameters, and execution controls (CPU)

Configure project-local caches before importing libraries. All training/evaluation branches are opt-in. Defaults preserve the original experiment: Qwen3-1.7B, 512 tokens, rank-16 LoRA, FP16, and the original smoke/full training arguments.


In [ ]:
import gc
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

PROJECT = Path(os.environ.get("TRUTHFULNESS_PROJECT", Path.cwd())).resolve()


def inside_project(path):
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT / path
    path = path.resolve()
    if not path.is_relative_to(PROJECT):
        raise ValueError(f"Path must stay inside {PROJECT}: {path}")
    return path


DATA_DIR = inside_project("data/halueval_qa_seed42")
HF_CACHE = inside_project("hf_cache")
HUB_CACHE = inside_project("hf_cache/hub")
RUN_DIR = inside_project("results/qwen3_halueval_seed42")
ADAPTER_DIR = inside_project("adapters/qwen3_halueval_seed42")
SMOKE_RUN_DIR = inside_project("results/training_smoke")
SMOKE_ADAPTER_DIR = inside_project("adapters/qwen3_halueval_smoke")
EVAL_DIR = inside_project("results/tuned_full")

# Set caches BEFORE importing Hugging Face libraries. This notebook is offline:
# model/tokenizer/benchmark files must already be staged in this project's cache.
cache_environment = {
    "HF_HOME": HF_CACHE,
    "HF_HUB_CACHE": HUB_CACHE,
    "HF_DATASETS_CACHE": inside_project("hf_cache/datasets"),
    "HF_MODULES_CACHE": inside_project("hf_cache/modules"),
    "HF_XET_CACHE": inside_project("hf_cache/xet"),
    "HF_TOKEN_PATH": inside_project("hf_cache/token"),
    "TORCH_HOME": inside_project("hf_cache/torch"),
    "XDG_CACHE_HOME": inside_project("hf_cache/xdg"),
    "TRITON_CACHE_DIR": inside_project("hf_cache/triton"),
    "TORCHINDUCTOR_CACHE_DIR": inside_project("hf_cache/torchinductor"),
    "TMPDIR": inside_project("tmp"),
}
for name, location in cache_environment.items():
    os.environ[name] = str(location)
    if name != "HF_TOKEN_PATH":
        location.mkdir(parents=True, exist_ok=True)
tempfile.tempdir = str(cache_environment["TMPDIR"])
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.chdir(PROJECT)

import torch
from datasets import load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_LENGTH = 512
SMOKE_EXAMPLES = 32

# Explicit controls: setup alone cannot start training or evaluation.
RUN_SMOKE_TEST = False
TRAINING_MODE = None  # None, "fresh", or "resume" (mutually exclusive).
RUN_TUNED_EVAL = False
if TRAINING_MODE not in (None, "fresh", "resume"):
    raise ValueError("TRAINING_MODE must be None, 'fresh', or 'resume'.")

LORA_PARAMETERS = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)
SMOKE_PARAMETERS = dict(
    max_steps=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)
FULL_PARAMETERS = dict(
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    report_to="none",
)
print("Project:", PROJECT)
print("Python:", sys.executable)
print("Controls:", RUN_SMOKE_TEST, TRAINING_MODE, RUN_TUNED_EVAL)


## B — Load prepared splits and tokenizer (CPU)

Requires the existing dataset and locally cached fast tokenizer/chat template. No dataset/model download or package installation occurs. The objective remains question → `right_answer`; `knowledge` and `hallucinated_answer` are not used.


In [ ]:
# Dependencies are prepared separately; this notebook never installs packages.
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Prepared HaluEval split missing: {DATA_DIR}. "
        "Prepare the seed-42 split with 01_importing_data.ipynb first."
    )
splits = load_from_disk(str(DATA_DIR))
for split_name in ("train", "validation"):
    if split_name not in splits or len(splits[split_name]) == 0:
        raise ValueError(f"Missing or empty split: {split_name}")
    missing = {"question", "right_answer"} - set(splits[split_name].column_names)
    if missing:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing)}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=str(HUB_CACHE),
    local_files_only=True,
    use_fast=True,
)
if not tokenizer.is_fast or not tokenizer.chat_template:
    raise ValueError("A fast Qwen3 tokenizer with its chat template is required.")
if tokenizer.pad_token is None:
    if tokenizer.eos_token is None:
        raise ValueError("Tokenizer has neither a pad token nor an EOS token.")
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Train:", len(splits["train"]), "| Validation:", len(splits["validation"]))
print("Tokenizer ready; no model weights or GPU tensors loaded.")


## C — Shared preprocessing and training helpers (CPU definitions only)

Masking now verifies the exact prompt prefix in text and tokens. The 512-token right truncation is unchanged, but examples retaining no actual answer tokens raise an error instead of silently training on invalid labels. No rows are dropped and no prompt is shortened. Template defaults, assistant continuation/end tokens, and the original objective are preserved. Fast-tokenizer offsets distinguish answer text from template scaffolding. Separate tokenization uses `add_special_tokens=False` as required by [Transformers chat-template guidance](https://github.com/huggingface/transformers/blob/main/docs/source/en/chat_templating.md).

Checkpoint validation requires saved adapter weights and Trainer/optimizer/scheduler/RNG state. References to a best checkpoint must remain inside this project and exist. Do not resume older runs with incompatible preprocessing or settings; a new experiment starts fresh.


In [ ]:
def require_allocated_cuda():
    if not os.environ.get("SLURM_JOB_ID"):
        raise RuntimeError("Run this step inside an existing Unity Slurm GPU allocation.")
    if not torch.cuda.is_available():
        raise RuntimeError("The selected kernel cannot use a CUDA GPU in this allocation.")


def require_empty_destination(path):
    path = inside_project(path)
    if path.exists() and (not path.is_dir() or any(path.iterdir())):
        raise FileExistsError(
            f"Refusing to overwrite existing results: {path}. "
            "Preserve the existing run before starting another."
        )


def tokenize_example(example, index):
    question = example["question"]
    answer = example["right_answer"]
    if not isinstance(question, str) or not question.strip():
        raise ValueError(f"Example {index}: missing question text.")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError(f"Example {index}: missing right_answer text.")
    user_text = "Answer truthfully and concisely.\n\n" + f"Question: {question}"
    prompt = [{"role": "user", "content": user_text}]
    conversation = prompt + [{"role": "assistant", "content": answer}]

    # Keep Qwen's existing template defaults, including thinking behavior.
    prompt_text = tokenizer.apply_chat_template(
        prompt, tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False
    )
    if not full_text.startswith(prompt_text):
        raise ValueError(f"Example {index}: prompt is not a prefix of the full chat.")
    # The template already supplies special tokens: do not add them again.
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    encoded = tokenizer(
        full_text, add_special_tokens=False, return_offsets_mapping=True
    )
    all_ids = encoded["input_ids"]
    if all_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(f"Example {index}: token boundary mismatch; refusing wrong labels.")

    input_ids = all_ids[:MAX_LENGTH]
    prompt_length = len(prompt_ids)
    labels = [-100] * min(prompt_length, len(input_ids))
    labels += input_ids[prompt_length:]

    # Check actual answer text, not just assistant scaffolding or an EOS token.
    answer_start = full_text.rfind(answer.strip())
    answer_end = answer_start + len(answer.strip())
    if answer_start < len(prompt_text):
        raise ValueError(f"Example {index}: cannot locate the assistant answer.")
    has_answer_token = any(
        label != -100 and end > answer_start and start < answer_end
        for label, (start, end) in zip(
            labels, encoded["offset_mapping"][:MAX_LENGTH]
        )
    )
    if not has_answer_token:
        raise ValueError(
            f"Example {index}: truncation at {MAX_LENGTH} tokens removes all answer "
            "supervision. Review this example; no rows are silently dropped."
        )
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def tokenize_split(name, limit=None):
    dataset = splits[name]
    if limit is not None:
        if len(dataset) < limit:
            raise ValueError(f"Smoke test requires {limit} training examples.")
        dataset = dataset.select(range(limit))
    # Rebuild labels after the masking fix instead of reusing stale map caches.
    return dataset.map(
        tokenize_example,
        with_indices=True,
        remove_columns=dataset.column_names,
        load_from_cache_file=False,
        desc=f"Tokenizing {name}",
    )


def latest_checkpoint():
    run_dir = inside_project(RUN_DIR)
    if not run_dir.is_dir():
        return None
    checkpoint = get_last_checkpoint(str(run_dir))
    return inside_project(checkpoint) if checkpoint is not None else None


def validate_checkpoint(checkpoint):
    checkpoint = inside_project(checkpoint)
    if checkpoint.parent != inside_project(RUN_DIR):
        raise ValueError("Resume checkpoint must be a direct child of RUN_DIR.")
    required = (
        "adapter_config.json", "trainer_state.json", "optimizer.pt",
        "scheduler.pt", "rng_state.pth",
    )
    missing = [name for name in required if not inside_project(checkpoint / name).is_file()]
    weights = ("adapter_model.safetensors", "adapter_model.bin")
    if not any(inside_project(checkpoint / name).is_file() for name in weights):
        missing.append("adapter_model.safetensors (or adapter_model.bin)")
    if missing:
        raise FileNotFoundError(f"Incomplete Trainer checkpoint {checkpoint}: {missing}")
    config = json.loads((checkpoint / "adapter_config.json").read_text())
    if config.get("base_model_name_or_path") != MODEL_ID:
        raise ValueError("Checkpoint base model does not match MODEL_ID.")
    state = json.loads((checkpoint / "trainer_state.json").read_text())
    best = state.get("best_model_checkpoint")
    if best:
        best = inside_project(best)
        if not best.is_dir() or not any(
            inside_project(best / name).is_file() for name in weights
        ):
            raise FileNotFoundError(f"Best-model checkpoint referenced by state is missing: {best}")
    return checkpoint


def run_lora_training(*, smoke=False, checkpoint=None):
    # All model/Trainer objects belong to this call; no previous run is needed.
    require_allocated_cuda()
    if smoke and checkpoint is not None:
        raise ValueError("Smoke testing cannot resume a full-training checkpoint.")
    run_dir = inside_project(SMOKE_RUN_DIR if smoke else RUN_DIR)
    adapter_dir = inside_project(SMOKE_ADAPTER_DIR if smoke else ADAPTER_DIR)
    if checkpoint is None:
        require_empty_destination(run_dir)
    else:
        checkpoint = validate_checkpoint(checkpoint)
    require_empty_destination(adapter_dir)
    model = trainer = None
    try:
        train_data = tokenize_split("train", SMOKE_EXAMPLES if smoke else None)
        val_data = None if smoke else tokenize_split("validation")
        torch.cuda.reset_peak_memory_stats()
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            cache_dir=str(HUB_CACHE),
            local_files_only=True,
        ).to("cuda")
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        model = get_peft_model(model, LoraConfig(**LORA_PARAMETERS))
        model.print_trainable_parameters()
        collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            padding=True,
            label_pad_token_id=-100,
            return_tensors="pt",
        )
        parameters = SMOKE_PARAMETERS if smoke else FULL_PARAMETERS
        args = TrainingArguments(output_dir=str(run_dir), **parameters)
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_data,
            eval_dataset=val_data,
            data_collator=collator,
            processing_class=tokenizer,
        )
        if checkpoint is None:
            trainer.train()
        else:
            trainer.train(resume_from_checkpoint=str(checkpoint))
        trainer.save_model(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))
        print("Adapter saved:", adapter_dir)
        print("Peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1024**3, 2))
    finally:
        trainer = None
        model = None
        gc.collect()
        torch.cuda.empty_cache()


print("Shared helpers ready. No tokenization or training has started.")


## D — Optional smoke test (GPU)

Enable `RUN_SMOKE_TEST` in A, rerun A–C, then this cell. Uses exactly the original 32-example subset, two optimizer steps, and smoke hyperparameters. Saves to `results/training_smoke` and `adapters/qwen3_halueval_smoke`; both destinations must be empty or absent. The following two comment-only cells describe historical results and the loss interpretation.


In [ ]:
if RUN_SMOKE_TEST:
    run_lora_training(smoke=True)
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST=True in the configuration to enable it.")


In [ ]:
#That’s a successful LoRA smoke test:
#- 17.4 million trainable parameters (1.003%) means LoRA is updating a small adapter while leaving the original model weights frozen.
#- Loss falling from 7.25 to 3.73 in two steps shows the training code is producing a learning signal. Two steps on 32 examples are far too little to judge model quality.
#- 3.61 GB peak GPU memory is a good sign for 11 GB GPU.

In [ ]:
#During fine-tuning, the model learns to predict the answer tokens for each question.
#- Training loss measures how wrong those predictions are on the 9,500 examples used to update the LoRA adapter. Lower means it is predicting those answers better.
#- Validation loss measures the same thing on the separate 500 examples that are not used to update the adapter. It helps us see whether learning carries over to unseen HaluEval examples.

## E — Fresh full training (GPU)

Choose `TRAINING_MODE="fresh"`, run A–C, then this cell. No smoke-test objects or checkpoints are needed. Trains a fresh adapter for one epoch with the original hyperparameters and selects the best saved model by validation loss. The run and final-adapter destinations must be empty or absent. This branch and G are mutually exclusive.


In [ ]:
if TRAINING_MODE == "fresh":
    # Starts from the base model with a NEW adapter; no checkpoint is required.
    run_lora_training()
else:
    print("Fresh training skipped. Set TRAINING_MODE='fresh' to enable it.")


## F — Optional checkpoint inspection (CPU)

Run after A–C to inspect an interrupted run. A missing run directory/checkpoint is reported without an exception or any new training. This inspection is not required for fresh training.


In [ ]:
# CPU-only inspection; safe when the run directory does not exist.
last_checkpoint = latest_checkpoint()
print("Latest checkpoint:", last_checkpoint if last_checkpoint else "none found")
print("No model or Trainer from an earlier session is required.")


## G — Optional checkpoint resumption (GPU when a checkpoint exists)

Choose `TRAINING_MODE="resume"`, run A–C, optionally F, then this cell. It rebuilds all model/Trainer objects and resumes the latest complete single-GPU Trainer checkpoint with the same full-training parameters. No checkpoint means a clear message and no training; incomplete checkpoints stop before loading a model. There is no automatic fallback to a fresh run.


In [ ]:
if TRAINING_MODE == "resume":
    # Discover again so this step does not depend on the inspection cell's state.
    checkpoint_to_resume = latest_checkpoint()
    if checkpoint_to_resume is None:
        print(
            "No checkpoint found; resumption skipped without starting training. "
            "For a new experiment, choose TRAINING_MODE='fresh'."
        )
    else:
        checkpoint_to_resume = validate_checkpoint(checkpoint_to_resume)
        print("Resuming from:", checkpoint_to_resume)
        run_lora_training(checkpoint=checkpoint_to_resume)
else:
    print("Resumption skipped. Set TRAINING_MODE='resume' only for an interrupted run.")


## H — Optional tuned evaluation (GPU)

After a final adapter is saved, enable `RUN_TUNED_EVAL`, run A–C, then this cell. It also works in a fresh allocated kernel without running D/E/G. Requires `lm_eval[hf]`, cached base-model/TruthfulQA files, and an empty or absent `results/tuned_full`. The existing method is unchanged: full TruthfulQA MC1/MC2, FP16, batch size 1, zero-shot, and logged samples. No smoke-test limit or chat-template override is added. Use [04_compare_eval.ipynb](04_compare_eval.ipynb) afterward; its hardcoded summary scores remain a separate unresolved issue.


In [ ]:
if RUN_TUNED_EVAL:
    require_allocated_cuda()
    adapter_dir = inside_project(ADAPTER_DIR)
    if not inside_project(adapter_dir / "adapter_config.json").is_file() or not any(
        inside_project(adapter_dir / name).is_file()
        for name in ("adapter_model.safetensors", "adapter_model.bin")
    ):
        raise FileNotFoundError(f"A complete final LoRA adapter is required: {adapter_dir}")
    config = json.loads((adapter_dir / "adapter_config.json").read_text())
    if config.get("base_model_name_or_path") != MODEL_ID:
        raise ValueError("Evaluation adapter base model does not match MODEL_ID.")
    out = inside_project(EVAL_DIR)
    require_empty_destination(out)
    out.mkdir(parents=True, exist_ok=True)
    # Same full TruthfulQA protocol as the baseline; no chat-template override.
    cmd = [
        sys.executable, "-m", "lm_eval", "run",
        "--model", "hf",
        "--model_args", f"pretrained={MODEL_ID},dtype=float16,peft={adapter_dir}",
        "--tasks", "truthfulqa_mc1,truthfulqa_mc2",
        "--device", "cuda:0",
        "--batch_size", "1",
        "--num_fewshot", "0",
        "--output_path", str(out),
        "--log_samples",
    ]
    subprocess.run(cmd, check=True, cwd=str(PROJECT), env=os.environ.copy())
else:
    print("Tuned evaluation skipped. Set RUN_TUNED_EVAL=True after saving a final adapter.")
